### ЗАДАЧА: Пакетная обработка переводов между кошельками (exceptions + business rules)

Есть набор кошельков и список входящих переводов.
Нужно безопасно обработать пакет операций: валидные переводы применить к балансам,
ошибочные сохранить в отчёт, не останавливая всю обработку.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `TransferError`
   - `TransferFormatError`
   - `AccountNotFoundError`
   - `CurrencyMismatchError`
   - `InsufficientFundsError`
   - `TransferAmountError`.

2. Функцию `parse_transfer(raw)`:
   - формат строки: `transfer_id|from_user|to_user|amount`
   - `amount` должен быть числом и `> 0`
   - при ошибке конвертации использовать `raise ... from ...`.

3. Функцию `apply_transfer(transfer, wallets)`:
   - проверить, что оба пользователя существуют
   - нельзя переводить самому себе
   - валюты кошельков отправителя и получателя должны совпадать
   - у отправителя должно хватать средств
   - при успехе обновить балансы в `wallets`
   - вернуть краткий словарь результата.

4. Функцию `process_batch(rows, wallets)`:
   - для каждой строки вызвать `parse_transfer`, потом `apply_transfer`
   - вернуть `(successes, errors)`
   - ошибки хранить как `(raw, error_type, message)`
   - не прерывать цикл на первой ошибке.

5. Вывести:
   - успешные переводы,
   - ошибки по типам,
   - итоговые балансы,
   - пользователя с максимальным балансом в валюте `USD`.


In [ ]:
wallets = {
    'alice': {'currency': 'USD', 'balance': 1200.0},
    'bob': {'currency': 'USD', 'balance': 450.0},
    'carol': {'currency': 'EUR', 'balance': 900.0},
    'dave': {'currency': 'USD', 'balance': 150.0},
}

rows = [
    'TR-100|alice|bob|200',
    'TR-101|bob|dave|700',
    'TR-102|alice|carol|50',
    'TR-103|eve|bob|30',
    'TR-104|dave|dave|10',
    'TR-105|bob|alice|abc',
    'TR-106|bob|dave|100',
]

class TransferError(Exception):
    pass

class TransferFormatError(TransferError):
    pass

class AccountNotFoundError(TransferError):
    pass

class CurrencyMismatchError(TransferError):
    pass

class InsufficientFundsError(TransferError):
    pass

class TransferAmountError(TransferError):
    pass


def parse_transfer(raw):
    """Парсит строку перевода и возвращает словарь с данными перевода."""
    try:
        parts = raw.split('|')
        if len(parts) != 4:
            raise TransferFormatError(f"Некорректный формат строки: ожидается 4 поля, получено {len(parts)}")

        transfer_id, sender, receiver, amount_str = parts

        try:
            amount = float(amount_str)
            if amount <= 0:
                raise TransferAmountError(f"Сумма перевода должна быть положительной: {amount}")
        except ValueError as e:
            raise TransferAmountError(f"Некорректное значение суммы: '{amount_str}'") from e

        return {
            'transfer_id': transfer_id,
            'sender': sender,
            'receiver': receiver,
            'amount': amount
        }

    except (TransferFormatError, TransferAmountError):
        raise
    except Exception as e:
        raise TransferFormatError(f"Неожиданная ошибка при парсинге строки: {raw}") from e



def apply_transfer(transfer, wallets):
    """Применяет перевод к кошелькам, проверяя все условия."""
    sender = transfer['sender']
    receiver = transfer['receiver']
    amount = transfer['amount']

    if sender not in wallets:
        raise AccountNotFoundError(f"Аккаунт отправителя не найден: '{sender}'")
    if receiver not in wallets:
        raise AccountNotFoundError(f"Аккаунт получателя не найден: '{receiver}'")

    if sender == receiver:
        raise TransferError(f"Запрещён перевод самому себе: '{sender}' → '{receiver}'")

    if wallets[sender]['currency'] != wallets[receiver]['currency']:
        raise CurrencyMismatchError(
            f"Несовпадение валют: {wallets[sender]['currency']} ≠ {wallets[receiver]['currency']}"
        )

    if wallets[sender]['balance'] < amount:
        raise InsufficientFundsError(
            f"Недостаточно средств: баланс {wallets[sender]['balance']}, требуется {amount}"
        )

    wallets[sender]['balance'] -= amount
    wallets[receiver]['balance'] += amount

    return {
        'transfer_id': transfer['transfer_id'],
        'sender': sender,
        'receiver': receiver,
        'amount': amount,
        'status': 'success'
    }



def process_batch(rows, wallets):
    """Обрабатывает пакет переводов, возвращая успешные операции и ошибки."""
    successes = []
    errors = []

    for i, row in enumerate(rows):
        try:
            transfer = parse_transfer(row)
            result = apply_transfer(transfer, wallets)
            successes.append(result)
        except TransferError as e:
            errors.append({
                'row_index': i,
                'row_content': row,
                'error_type': type(e).__name__,
                'error_message': str(e)
            })
        except Exception as e:
            errors.append({
                'row_index': i,
                'row_content': row,
                'error_type': 'UnexpectedError',
                'error_message': f"Неожиданная ошибка: {e}"
            })

    return successes, errors

successes, errors = process_batch(rows, wallets)

print("Успешные переводы:")
if successes:
    for success in successes:
        print(f"  {success['transfer_id']}: {success['sender']} → {success['receiver']} | {success['amount']}")
else:
    print("  Нет успешных переводов")

print("\nОшибки по типам:")
if errors:
    error_counts = {}
    for error in errors:
        error_type = error['error_type']
        error_counts[error_type] = error_counts.get(error_type, 0) + 1

    for error_type, count in error_counts.items():
        print(f"  {error_type}: {count} ошибок")

    print("\nДетали ошибок:")
    for error in errors:
        print(f"  Строка {error['row_index']}: {error['error_message']} (содержимое: '{error['row_content']}')")
else:
    print("  Ошибок нет.")

print("\nИтоговые балансы:")
for user, data in wallets.items():
    print(f"  {user}: {data['balance']} {data['currency']}")

usd_users = {user: data['balance'] for user, data in wallets.items() if data['currency'] == 'USD'}
if usd_users:
    richest_usd_user = max(usd_users, key=usd_users.get)
    richest_balance = usd_users[richest_usd_user]
    print(f"\nСамый большой балланс в USD: {richest_usd_user} ({richest_balance} USD)")
else:
    print("\nПользователей с валютой USD нет.")
